In [2]:
import pandas as pd
import json
import os
from google.colab import drive
from google.colab import files

# ==========================================
# 1. SETUP & FILE LOADING
# ==========================================

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Define input/output names
# We assume the file is named similarly to your others
filename = '/content/drive/MyDrive/Imperative sentence - Sheet1 (1).csv'
output_filename = 'nllb_multilingual_imperative_dataset.json'

# Paths to check automatically
possible_paths = [
    f'/content/{filename}',                        # Direct upload
    f'/content/drive/My Drive/{filename}',         # Root of Drive
    f'/content/drive/MyDrive/{filename}',          # Root of Drive (alternate)
    f'/content/drive/MyDrive/datasets/{filename}', # Common datasets folder
]

input_path = None

# Check if file exists in Drive or local storage
for path in possible_paths:
    if os.path.exists(path):
        input_path = path
        print(f"✅ Found file at: {input_path}")
        break

# If not found, ask user to upload it NOW
if input_path is None:
    print(f"⚠️ File '{filename}' not found automatically.")
    print("Please upload your Imperative CSV file now using the button below:")
    uploaded = files.upload()
    if uploaded:
        input_path = list(uploaded.keys())[0] # Get the name of the uploaded file
        print(f"✅ File uploaded: {input_path}")
    else:
        print("❌ No file uploaded. Please re-run the cell and upload the file.")

# ==========================================
# 2. DATA CLEANING FUNCTIONS (IMPERATIVE SPECIFIC)
# ==========================================

def clean_text_imperative(text, script_type):
    """
    Cleans text for Imperative sentences.
    - Removes question marks '?' (since commands aren't questions).
    - Ensures it ends with '.' (English/Latn) or '।' (Bangla/Beng).
    - Preserves exclamation marks '!' as they are valid for commands.
    """
    if not isinstance(text, str):
        return ""

    text = text.strip()
    if not text:
        return ""

    # Danda character for Bangla
    danda = '।'

    # 1. Remove Question Marks (Imperative shouldn't be a question)
    if text.endswith('?'):
        text = text[:-1].strip()

    # 2. Add Punctuation based on script
    if script_type == 'latn':
        # English/Marma-Latn: Should end in '.' or '!'
        if text and text[-1] not in ['.', '!']:
            text += '.'

    elif script_type == 'beng':
        # Bangla/Marma-Beng: Should end in '|' or '!'
        # Note: We check for '.' too just in case it was typed by mistake,
        # but strictly for Bangla script we usually want danda.
        if text and text[-1] not in [danda, '!', '.']:
            text += danda

    return text

# ==========================================
# 3. PROCESSING & AUGMENTATION
# ==========================================

if input_path:
    try:
        print("\nReading CSV file...")
        df = pd.read_csv(input_path)

        # Validate columns exist
        required_cols = ['English', 'Bangla', 'Marma_English_Transliterate', 'Marma_Bangla_Transliterate']
        if not all(col in df.columns for col in required_cols):
            raise ValueError(f"CSV is missing columns.\nFound: {df.columns}\nNeeded: {required_cols}")

        print(f"Processing {len(df)} rows...")
        print("Cleaning punctuation (Enforcing '.' and '|' for commands)...")

        # Apply specific IMPERATIVE cleaning
        # English & Marma-English -> Use Latin rules (add .)
        df['English'] = df['English'].apply(lambda x: clean_text_imperative(x, 'latn'))
        df['Marma_English_Transliterate'] = df['Marma_English_Transliterate'].apply(lambda x: clean_text_imperative(x, 'latn'))

        # Bangla & Marma-Bangla -> Use Bangla rules (add |)
        df['Bangla'] = df['Bangla'].apply(lambda x: clean_text_imperative(x, 'beng'))
        df['Marma_Bangla_Transliterate'] = df['Marma_Bangla_Transliterate'].apply(lambda x: clean_text_imperative(x, 'beng'))

        print("Augmenting data (Generating All Pairs)...")
        augmented_dataset = []

        # Iterate through rows and generate permutations
        for index, row in df.iterrows():
            # Map text to language codes
            texts = {
                'eng_Latn': row['English'],
                'ben_Beng': row['Bangla'],
                'mrm_Latn': row['Marma_English_Transliterate'],
                'mrm_Beng': row['Marma_Bangla_Transliterate']
            }

            # Generate every possible pair (Source -> Target)
            keys = list(texts.keys())
            for src_lang in keys:
                for tgt_lang in keys:
                    if src_lang != tgt_lang:
                        # Skip if either source or target is empty
                        if not texts[src_lang] or not texts[tgt_lang]:
                            continue

                        # NLLB / Unsloth JSON Structure
                        sample = {
                            "translation": {
                                "src": texts[src_lang],
                                "tgt": texts[tgt_lang],
                                "src_lang": src_lang,
                                "tgt_lang": tgt_lang
                            }
                        }
                        augmented_dataset.append(sample)

        # ==========================================
        # 4. SAVING TO DRIVE
        # ==========================================

        # Define save path in Drive
        save_path = f'/content/drive/My Drive/{output_filename}'

        print(f"Saving {len(augmented_dataset)} translation pairs to Google Drive...")
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(augmented_dataset, f, ensure_ascii=False, indent=2)

        print("\n✅ SUCCESS!")
        print(f"Dataset saved correctly at: {save_path}")
        print(f"Check your Google Drive folder for '{output_filename}'")

    except Exception as e:
        print(f"\n❌ An error occurred during processing: {e}")

⚠️ File '/content/drive/MyDrive/Imperative sentence - Sheet1 (1).csv' not found automatically.
Please upload your Imperative CSV file now using the button below:


Saving Imperative sentence - Sheet1 (1).csv to Imperative sentence - Sheet1 (1).csv
✅ File uploaded: Imperative sentence - Sheet1 (1).csv

Reading CSV file...
Processing 1649 rows...
Cleaning punctuation (Enforcing '.' and '|' for commands)...
Augmenting data (Generating All Pairs)...
Saving 19692 translation pairs to Google Drive...

✅ SUCCESS!
Dataset saved correctly at: /content/drive/My Drive/nllb_multilingual_imperative_dataset.json
Check your Google Drive folder for 'nllb_multilingual_imperative_dataset.json'
